Here we use SVI to sample

Define the model and create data in the same way. Be careful of the leading n_simulations and num_samples axis. 

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
from numpyro.infer import Predictive

import dynestyx as dsx
from dynestyx import DiscreteTimeSimulator, DynamicalModel

# for convenience, we can define "fixed" things in the model outside of it.
# this is not required, but it helps keep the model clean.
state_dim = 2
observation_dim = 1
control_dim = 1

# Create the known matrices B, C
B = jnp.eye(state_dim, control_dim)
C = jnp.eye(observation_dim, state_dim)

# create the initial condition as a distribution
initial_condition = dist.MultivariateNormal(jnp.zeros(state_dim), jnp.eye(state_dim))


def lti_model(
    sigma_obs=0.1,
    sigma_process=0.1,
    obs_times=None,
    obs_values=None,
    ctrl_times=None,
    ctrl_values=None,
    predict_times=None,
):
    # sample the unknown parameter
    rho = numpyro.sample("rho", dist.Uniform(-0.5, 0.5))
    A = jnp.array([[0, 0.3], [rho, -0.2]])

    # create the state evolution as a callable mapping to a distribution
    # Crucially, this depends on A, which depends on rho, which is unknown.
    # Thus, the state evolution MUST be defined within `lti_model`, not outside.
    state_evolution = lambda x, u, t_now, t_next: dist.MultivariateNormal(
        A @ x + B @ u, sigma_process**2 * jnp.eye(state_dim)
    )

    # create the observation model as a callable mapping to a distribution
    observation_model = lambda x, u, t: dist.MultivariateNormal(
        C @ x, sigma_obs**2 * jnp.eye(observation_dim)
    )

    # create the dynamical model
    dynamics = DynamicalModel(
        control_dim=control_dim,
        initial_condition=initial_condition,
        state_evolution=state_evolution,
        observation_model=observation_model,
    )

    # sample from the dynamical model
    return dsx.sample(
        "f",
        dynamics,
        obs_times=obs_times,
        obs_values=obs_values,
        ctrl_times=ctrl_times,
        ctrl_values=ctrl_values,
        predict_times=predict_times,
    )

    # create a synthetic control sequence as i.i.d. Gaussians
obs_times = jnp.arange(0.0, 100.0, 1.0)  # T=100 steps
ctrl_times = obs_times  # same times for controls
ctrl_values = jr.normal(jr.PRNGKey(0), (len(ctrl_times), control_dim))
rho_true = 0.3


def make_data(sigma_obs=0.1, sigma_process=0.1):
    predictive = Predictive(
        lti_model,
        params={"rho": jnp.array(rho_true)},
        num_samples=1,
        exclude_deterministic=False,
    )
    with DiscreteTimeSimulator():
        pred = predictive(
            rng_key=jr.PRNGKey(0),
            sigma_obs=sigma_obs,
            sigma_process=sigma_process,
            predict_times=obs_times,
            ctrl_times=ctrl_times,
            ctrl_values=ctrl_values,
        )
    print("make_data shapes:", pred["f_times"].shape, pred["f_observations"].shape)
    # Expected: f_observations has shape (1, 1, T, obs_dim)
    obs_values = pred["f_observations"][0, 0, :, :]
    return obs_times, obs_values, ctrl_times, ctrl_values


obs_times, obs_values, ctrl_times, ctrl_values = make_data(sigma_obs=0.1, sigma_process=0.1)

Running Inference with SVI

In [ ]:
import optax
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoMultivariateNormal

from dynestyx import Filter
num_svi_steps = 1500
num_posterior_samples = 500


# Filter-based SVI (filter-based; record filtered states for latent recovery plot)
#Shorthand for the model within a Filter context. Note that the model itself is unchanged, but the inference algorithm will be different when we run SVI on it.
def filter_conditioned_model():
    # Useful shorthand, since we're going to use `with Filter()` several times!
    # Uses EnKF by default
    with Filter():
        return lti_model(obs_times=obs_times, obs_values=obs_values, ctrl_times=ctrl_times, ctrl_values=ctrl_values)

#Establish the guide automultivariate normal, argument is the model we want to approximate 
guide1 = AutoMultivariateNormal(filter_conditioned_model)
optimizer = optax.adam(learning_rate=1e-3)
svi1 = SVI(filter_conditioned_model, guide1, optimizer, loss=Trace_ELBO())
svi_result1 = svi1.run(jr.PRNGKey(1), num_steps=num_svi_steps)
posterior_1 = Predictive(
    guide1, params=svi_result1.params, num_samples=num_posterior_samples
)(jr.PRNGKey(2))


# Joint state + parameter SVI
def joint_conditioned_model():
    # Useful shorthand, since we're going to use `with DiscreteTimeSimulator()` several times!
    with DiscreteTimeSimulator():
        return lti_model(obs_times=obs_times, obs_values=obs_values, ctrl_times=ctrl_times, ctrl_values=ctrl_values)


guide2 = AutoMultivariateNormal(joint_conditioned_model)
svi2 = SVI(joint_conditioned_model, guide2, optimizer, loss=Trace_ELBO())
#Times 10 since it takes longer to learn
svi_result2 = svi2.run(jr.PRNGKey(3), num_steps=int(10 * num_svi_steps))
posterior_2_raw = Predictive(
    guide2, params=svi_result2.params, num_samples=num_posterior_samples
)(jr.PRNGKey(4))
T = len(obs_times)
posterior_2 = {"rho": posterior_2_raw["rho"]}
# _auto_latent is (num_samples, 1 + T*state_dim); first element is rho, rest are states
latent = posterior_2_raw["_auto_latent"]
posterior_2["f_states"] = latent[:, 1:].reshape(num_posterior_samples, T, state_dim)

Plot the inferences

In [ ]:
import arviz as az
import matplotlib.pyplot as plt

az.plot_posterior(posterior_1["rho"], hdi_prob=0.95, ref_val=rho_true)
plt.title("SVI + KF: parameter inference with filter-based marginalization")

az.plot_posterior(posterior_2["rho"], hdi_prob=0.95, ref_val=rho_true)
plt.title("SVI: joint state + parameter inference")
plt.show()

Compare latent state recoveries